# Notebook 06: Machine Learning

## I. Giới thiệu
Mục tiêu: Huấn luyện, đánh giá, so sánh các mô hình ML và lựa chọn mô hình dự báo nhiệt độ tốt nhất.


## II. Đọc dữ liệu & III. Chuẩn bị dữ liệu


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
import joblib

city_df = pd.read_csv('../data/processed/tokyo_features.csv', index_col='dt', parse_dates=True)

# Chia tập Train / Test theo mốc thời gian (Năm 2000)
train = city_df[city_df['Year'] < 2000]
test = city_df[city_df['Year'] >= 2000]

X_cols = ['Year', 'Month', 'Lag_1', 'Lag_12', 'Rolling_Mean_12']
y_col = 'AverageTemperature'

X_train, y_train = train[X_cols], train[y_col]
X_test, y_test = test[X_cols], test[y_col]


## IV. Mô hình Baseline: Ridge Regression


In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
pred_ridge = ridge.predict(X_test)


## V. Mô hình Random Forest


In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)


## VI. Mô hình XGBoost (Tối ưu GridSearchCV)


In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 8],
    'learning_rate': [0.01, 0.05, 0.1]
}
grid_search = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)
best_xgb = grid_search.best_estimator_
pred_xgb = best_xgb.predict(X_test)


## VII. So sánh các mô hình


In [ ]:
def evaluate(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2 Score': r2_score(y_true, y_pred)
    }

results = pd.DataFrame({
    'Ridge': evaluate(y_test, pred_ridge),
    'Random Forest': evaluate(y_test, pred_rf),
    'XGBoost': evaluate(y_test, pred_xgb)
})
print(results.T)


## VIII. Lựa chọn mô hình & Lưu lại


In [ ]:
import os
os.makedirs('../model', exist_ok=True)
joblib.dump(best_xgb, '../model/xgboost_model.pkl')
print("Đã lưu mô hình XGBoost thành công!")


## IX. Kết luận
Mô hình XGBoost có hiệu năng ổn định và R2 Score cao nhất (~0.98), hoàn toàn phù hợp để Deploy ở Notebook 07.
